---
title: "Why Can't We Just Learn Label -> Image Directly?"
author: "Hujie Wang"
format: html
---

# Why Can't We Just Learn Label -> Image Directly?

This notebook demonstrates **why** we need diffusion/flow matching instead of direct regression.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from torchdiffeq import odeint

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## The Problem: One Input → Many Valid Outputs

Imagine a robot arm needs to reach around an obstacle. There are **two valid paths**:
- Go LEFT around the obstacle
- Go RIGHT around the obstacle

We'll simulate this as a **bimodal distribution** in 2D:
- Input: a single label (or condition)
- Output: a point that should be at (-3, 0) OR (+3, 0)

In [ ]:
# Our "data distribution": two valid modes
def sample_bimodal(n_samples):
    """Sample from two Gaussians centered at (-3, 0) and (+3, 0)"""
    # Randomly choose left or right mode
    left_mask = torch.rand(n_samples) < 0.5
    
    samples = torch.randn(n_samples, 2) * 0.5  # Small noise
    samples[left_mask, 0] -= 3   # Left mode at x=-3
    samples[~left_mask, 0] += 3  # Right mode at x=+3
    
    return samples

# Visualize the target distribution
data = sample_bimodal(1000)
plt.figure(figsize=(8, 6))
plt.scatter(data[:, 0], data[:, 1], alpha=0.5, s=10)
plt.axvline(x=0, color='red', linestyle='--', label='The forbidden zone (obstacle)')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Target Distribution: Two Valid Modes\n(Go left OR go right, but NOT through the middle!)')
plt.legend()
plt.xlim(-6, 6)
plt.ylim(-3, 3)
plt.show()

## Approach 1: Direct Regression (The Naive Way)

Let's try to learn a direct mapping: `noise z → output x`

We'll train with MSE loss: minimize `||f(z) - x_target||²`

In [ ]:
# A simple MLP that tries to map noise → data directly
class DirectRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )
    
    def forward(self, z):
        return self.net(z)

# Train direct regressor
direct_model = DirectRegressor().to(device)
optimizer = torch.optim.Adam(direct_model.parameters(), lr=1e-3)

losses = []
for step in range(2000):
    # Sample noise (input) and data (target)
    z = torch.randn(256, 2).to(device)  # Random noise input
    x_target = sample_bimodal(256).to(device)  # Target from bimodal dist
    
    # Forward pass
    x_pred = direct_model(z)
    
    # MSE loss - THIS IS THE PROBLEM!
    loss = ((x_pred - x_target) ** 2).mean()
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Final loss: {losses[-1]:.4f}")

In [ ]:
# Generate samples from the direct regressor
with torch.no_grad():
    z_test = torch.randn(500, 2).to(device)
    direct_samples = direct_model(z_test).cpu()

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: True data
axes[0].scatter(data[:, 0], data[:, 1], alpha=0.5, s=10, label='True data')
axes[0].axvline(x=0, color='red', linestyle='--')
axes[0].set_title('True Distribution')
axes[0].set_xlim(-6, 6)
axes[0].set_ylim(-3, 3)

# Right: Direct regression samples
axes[1].scatter(direct_samples[:, 0], direct_samples[:, 1], alpha=0.5, s=10, c='orange', label='Generated')
axes[1].axvline(x=0, color='red', linestyle='--')
axes[1].scatter([0], [0], s=200, c='red', marker='X', zorder=5, label='Mode collapse!')
axes[1].set_title('Direct Regression: MODE AVERAGING DISASTER!\nAll samples collapse to the MEAN (0, 0)')
axes[1].set_xlim(-6, 6)
axes[1].set_ylim(-3, 3)
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("THE PROBLEM: Direct regression outputs the AVERAGE of all modes!")
print("The robot would crash THROUGH the obstacle instead of going around!")
print("="*60)

## Why Does This Happen?

MSE loss minimizes: $\mathbb{E}[\|f(z) - x\|^2]$

The optimal solution is: $f^*(z) = \mathbb{E}[x]$ -- **the mean of all possible outputs!**

When outputs are multimodal (multiple valid answers), the mean falls **between** the modes, which is often **invalid**.

---

## Approach 2: Flow Matching (The Right Way)

Instead of learning `z -> x` directly, we learn the **vector field** that transports noise to data.

Key insight: Given a noisy point $x_t$ at time $t$, there's a **unique optimal direction** to move!

In [ ]:
# Flow matching model: predicts vector field given (x_t, t)
class VectorField(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 128),  # 2D point + time
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 2)   # 2D vector field output
        )
    
    def forward(self, x, t):
        # t is scalar or batch, expand to match x
        if t.dim() == 0:
            t = t.expand(x.shape[0])
        t = t.unsqueeze(-1)  # [B, 1]
        xt = torch.cat([x, t], dim=-1)  # [B, 3]
        return self.net(xt)

# Train flow matching
flow_model = VectorField().to(device)
optimizer = torch.optim.Adam(flow_model.parameters(), lr=1e-3)

losses = []
for step in range(5000):
    # Sample noise x0 and data x1
    x0 = torch.randn(256, 2).to(device)       # Noise
    x1 = sample_bimodal(256).to(device)       # Data
    
    # Sample random time
    t = torch.rand(256).to(device)
    
    # Interpolate: x_t = (1-t)*x0 + t*x1
    t_expand = t.unsqueeze(-1)
    xt = (1 - t_expand) * x0 + t_expand * x1
    
    # Target vector field: just the direction from x0 to x1!
    target_u = x1 - x0
    
    # Predict vector field
    pred_u = flow_model(xt, t)
    
    # MSE on vector field prediction
    loss = ((pred_u - target_u) ** 2).mean()
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Final loss: {losses[-1]:.4f}")

In [ ]:
# Sample from flow matching by integrating the ODE
@torch.no_grad()
def sample_flow(model, n_samples, n_steps=100):
    # Start from noise
    x = torch.randn(n_samples, 2).to(device)
    
    # Euler integration from t=0 to t=1
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = torch.tensor(i / n_steps).to(device)
        u = model(x, t)
        x = x + u * dt
    
    return x.cpu()

# Generate samples
flow_samples = sample_flow(flow_model, 500)

In [ ]:
# Compare all three: True data, Direct regression, Flow matching
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# True data
axes[0].scatter(data[:, 0], data[:, 1], alpha=0.5, s=10)
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('True Distribution\n(Two valid modes)', fontsize=12)
axes[0].set_xlim(-6, 6)
axes[0].set_ylim(-3, 3)

# Direct regression (FAIL)
axes[1].scatter(direct_samples[:, 0], direct_samples[:, 1], alpha=0.5, s=10, c='orange')
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Direct Regression\n(Collapsed to mean = INVALID)', fontsize=12)
axes[1].set_xlim(-6, 6)
axes[1].set_ylim(-3, 3)

# Flow matching (SUCCESS)
axes[2].scatter(flow_samples[:, 0], flow_samples[:, 1], alpha=0.5, s=10, c='green')
axes[2].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[2].set_title('Flow Matching\n(Both modes captured!)', fontsize=12)
axes[2].set_xlim(-6, 6)
axes[2].set_ylim(-3, 3)

plt.tight_layout()
plt.show()

## Why Does Flow Matching Work?

The key insight: **the vector field prediction problem has a unique answer!**

Even though `x1` (the target) can be multimodal, the expected vector field given `x_t` is:

$$u_\theta(x_t, t) \approx \mathbb{E}[x_1 - x_0 \mid x_t]$$

This IS well-defined because:
- Points near the LEFT mode get vector field pointing LEFT
- Points near the RIGHT mode get vector field pointing RIGHT
- The model learns to **separate** the modes based on spatial position!

In [ ]:
# Visualize the learned vector field
@torch.no_grad()
def plot_vector_field(model, t_val, ax, title):
    # Create grid
    x_range = torch.linspace(-5, 5, 20)
    y_range = torch.linspace(-2, 2, 10)
    xx, yy = torch.meshgrid(x_range, y_range, indexing='ij')
    points = torch.stack([xx.flatten(), yy.flatten()], dim=-1).to(device)
    
    # Get vector field values
    t = torch.tensor(t_val).to(device)
    u = model(points, t).cpu()
    
    # Plot
    ax.quiver(points[:, 0].cpu(), points[:, 1].cpu(), 
              u[:, 0], u[:, 1], scale=50, alpha=0.7)
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.3)
    ax.set_xlim(-5, 5)
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(title)

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for i, t_val in enumerate([0.0, 0.3, 0.6, 0.9]):
    plot_vector_field(flow_model, t_val, axes[i], f't = {t_val}')

plt.suptitle('Learned Vector Field at Different Times\n(Arrows show which direction points should move)', fontsize=12)
plt.tight_layout()
plt.show()

print("Notice: Points on the LEFT get pushed LEFT, points on the RIGHT get pushed RIGHT!")
print("The vector field naturally separates the modes.")

## The Robotics Connection

This is EXACTLY why diffusion/flow matching is popular in robotics:

| Scenario | Direct Policy | Diffusion Policy |
|----------|--------------|------------------|
| Reach around obstacle | Averages left+right = **crash into obstacle** | Samples either left OR right = **valid path** |
| Pick up object | Averages all grasp angles = **invalid grasp** | Samples one valid grasp = **success** |
| Navigate to goal | Averages all routes = **stuck** | Samples one route = **reaches goal** |

The "multimodality" means: **there are multiple valid actions, and we need to pick ONE, not average them all.**

---

## Summary: What Does $u_\theta(x_t, t)$ Learn?

It learns a **vector field** that answers:

> "Given this noisy/intermediate point $x_t$ at time $t$, which direction should I move to reach valid data?"

The magic:
1. **Stochasticity from initial noise**: Different starting points -> different final samples
2. **Deterministic vector field**: Given $(x_t, t)$, there's ONE optimal direction
3. **Mode separation**: The spatial structure of $x_t$ naturally encodes which mode to target

We're NOT learning the impossible mapping `input -> multimodal output`.

We're learning the tractable mapping `(noisy state, time) -> direction to go`.